Uses Stable Diffusion Inpainting to transform the image


In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload your 'composite.png'


In [ ]:
#No Inpainting – Use Text-to-Image with Image as Initial Condition (img2img)
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image
import torch

# Step 1: Device & Precision
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# Step 2: Load pipeline
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype
).to(device)

# Step 3: Load your image from disk
image_path = "/content/055.jpg"  # <-- change to your image filename
init_image = Image.open(image_path).convert("RGB").resize((512, 512))

# Step 4: Prompt and inference
prompt = "a foggy forest, cinematic lighting, dreamy atmosphere"
result = pipe(
    prompt=prompt,
    image=init_image,
    strength=0.75,        # 0.0 = preserve original, 1.0 = fully stylized
    guidance_scale=7.5
).images[0]

# Step 5: Save result locally
output_path = "/content/stylized_image.png"
result.save(output_path)
print(f"Saved to: {output_path}")



Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/37 [00:00<?, ?it/s]

Saved to: /content/stylized_image.png


In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw
import torch
import os
import shutil

# --- Settings ---
image_path = "/content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/train_img/image_77.jpg"
yolo_label_path = "/content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/train_txt/image_77.txt"
prompt = "restore missing part with shiny metallic silver robotic gripper in cinematic lighting"
output_path = "/content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/traininpaint_img/inimage_77.jpg"

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# --- Load pipeline ---
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-inpainting", torch_dtype=dtype
).to(device)

# --- Load and resize image to 512x512 ---
image = Image.open(image_path).convert("RGB").resize((512, 512))

# --- Create mask ---
mask = Image.new("L", (512, 512), color=255)
draw = ImageDraw.Draw(mask)

# Parse YOLO label and draw rectangles
with open(yolo_label_path, "r") as f:
    for line in f:
        cls, x_c, y_c, w, h = map(float, line.strip().split())
        x = (x_c - w / 2) * 512
        y = (y_c - h / 2) * 512
        w *= 512
        h *= 512
        draw.rectangle([x, y, x + w, y + h], fill=0)  # black = keep unchanged

# --- Run inpainting ---
result = pipe(
    prompt=prompt,
    image=image,
    mask_image=mask,
    guidance_scale=7.5,
    strength=1.0,
).images[0]

# --- Save inpainted image ---
result.save(output_path)
print(f"✅ Saved inpainted image to: {output_path}")

# --- Save corresponding YOLO .txt ---
txt_output_path = os.path.splitext(output_path)[0] + ".txt"

# Copy original YOLO labels to match new image
shutil.copy(yolo_label_path, txt_output_path)
print(f"✅ Saved YOLO-format label to: {txt_output_path}")


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

✅ Saved inpainted image to: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/traininpaint_img/inimage_77.jpg
✅ Saved YOLO-format label to: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/traininpaint_img/inimage_77.txt


In [ ]:
# SDXL Inpainting Batch Script (1024x1024)
# Requirements:
#   pip install diffusers accelerate transformers safetensors --upgrade
# Run in Colab / GPU environment with sufficient VRAM.

import os
import shutil
from PIL import Image, ImageDraw
from tqdm import tqdm
import torch

# Diffusers imports (SDXL inpaint pipeline)
from diffusers import StableDiffusionXLInpaintPipeline, StableDiffusionInpaintPipeline
from diffusers.utils import is_accelerate_available

# -----------------------
# Settings (edit paths)
# -----------------------
base_dir = "/content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label"
input_img_dir = os.path.join(base_dir, "train")
input_txt_dir = os.path.join(base_dir, "train_txt")
output_dir = os.path.join(base_dir, "real_inpaint_SDXL")
prompt = "Restore issing part with a shiny metallic silver texture under cinematic lighting"

os.makedirs(output_dir, exist_ok=True)

# -----------------------
# Device & dtype
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
# SDXL benefits from float16 on CUDA
dtype = torch.float16 if (device == "cuda") else torch.float32

# -----------------------
# Model loader helper
# -----------------------
def load_sdxl_inpaint_model(device, dtype):
    """
    Attempts to load SDXL inpainting model. Raises on failure.
    """
    model_id = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
    print(f"⏳ Loading SDXL inpaint model: {model_id}")
    pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
        # safety_checker can be None or left default; you can customize
    )
    # Move to device and enable memory optimizations if available
    pipe = pipe.to(device)
    # If using CUDA, consider enable_xformers_memory_efficient_attention or other settings
    try:
        # optional: enable memory-efficient attention if available
        if device == "cuda":
            try:
                pipe.enable_xformers_memory_efficient_attention()
            except Exception:
                pass
    except Exception:
        pass
    print("✅ SDXL inpaint pipeline loaded.")
    return pipe

# -----------------------
# Fallback loader (SD 1.5 inpaint)
# -----------------------
def load_sd15_inpaint_model(device, dtype):
    model_id = "runwayml/stable-diffusion-inpainting"
    print(f"⏳ Loading fallback SD1.5 inpaint model: {model_id}")
    pipe = StableDiffusionInpaintPipeline.from_pretrained(
        model_id,
        torch_dtype=dtype,
    ).to(device)
    print("✅ SD1.5 inpaint pipeline loaded.")
    return pipe

# -----------------------
# Try to load SDXL, otherwise fallback to SD1.5
# -----------------------
try:
    pipe = load_sdxl_inpaint_model(device, dtype)
    using_sdxl = True
except Exception as e:
    print("⚠️ Could not load SDXL inpaint model (will fallback to SD1.5).")
    print("Error:", e)
    pipe = load_sd15_inpaint_model(device, dtype)
    using_sdxl = False

# -----------------------
# Inference params (tweakable)
# -----------------------
if using_sdxl:
    target_size = (1024, 1024)      # SDXL native
    strength = 0.9999               # SDXL-specific recommended
    guidance_scale = 7.5
    num_inference_steps = 28
else:
    target_size = (512, 512)        # SD1.5 native
    strength = 1.0
    guidance_scale = 7.5
    num_inference_steps = 50

# -----------------------
# Helper: create mask from YOLO txt file
# -----------------------
def create_mask_from_yolo(label_path, size):
    """
    Returns a PIL 'L' mask resized to size.
    White (255) = inpaint (edit); Black (0) = keep original.
    """
    w, h = size
    # Start all black (keep original)
    mask = Image.new("L", (w, h), color=0)
    draw = ImageDraw.Draw(mask)
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, x_c, y_c, bw, bh = map(float, parts[:5])
            x_min = (x_c - bw / 2) * w
            y_min = (y_c - bh / 2) * h
            x_max = (x_c + bw / 2) * w
            y_max = (y_c + bh / 2) * h
            # Fill white area to inpaint
            draw.rectangle([x_min, y_min, x_max, y_max], fill=255)
    return mask

# -----------------------
# Batch processing
# -----------------------
image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"🔍 Found {len(image_files)} images to process. Using SDXL: {using_sdxl}")

for idx, image_name in enumerate(sorted(image_files), start=1):
    img_path = os.path.join(input_img_dir, image_name)
    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(input_txt_dir, label_name)

    if not os.path.exists(label_path):
        print(f"⚠️ Skipping {image_name}: label not found.")
        continue

    # Load original and resize to model native
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"⚠️ Could not open {image_name}: {e}")
        continue

    image_resized = image.resize(target_size, resample=Image.LANCZOS)
    mask = create_mask_from_yolo(label_path, target_size)

    # Run pipeline
    prompt_for_run = prompt
    # SDXL pipeline call signature may differ; both SDXL and SD1.5 pipelines accept these args
    try:
        output = pipe(
            prompt=prompt_for_run,
            image=image_resized,
            mask_image=mask,
            strength=strength,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
        )
        # Get resulting PIL image
        result_img = output.images[0]
    except TypeError:
        # older/newer diffusers versions may name args differently; try without strength if error
        output = pipe(
            prompt=prompt_for_run,
            image=image_resized,
            mask_image=mask,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
        )
        result_img = output.images[0]
    except Exception as e:
        print(f"❌ Inpainting failed on {image_name}: {e}")
        continue

    # Save at native resolution
    output_name = "sdxl_in_" + image_name
    output_path = os.path.join(output_dir, output_name)
    result_img.save(output_path)
    # Copy label (keeps original YOLO file for tracking)
    shutil.copy(label_path, os.path.splitext(output_path)[0] + ".txt")

    print(f"[{idx}/{len(image_files)}] ✅ Saved: {output_path}")

print("🎉 Batch inpainting complete. Results in:", output_dir)


⏳ Loading SDXL inpaint model: diffusers/stable-diffusion-xl-1.0-inpainting-0.1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

... (19 outputs left out)

[2/350] ✅ Saved: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/real_inpaint_SDXL/sdxl_in_054.jpg


  0%|          | 0/27 [00:00<?, ?it/s]

[3/350] ✅ Saved: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/real_inpaint_SDXL/sdxl_in_055.jpg


  0%|          | 0/27 [00:00<?, ?it/s]

In [5]:
# --- Batch Inpainting Using SD 1.5 ---
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image, ImageDraw
import torch
import os
import shutil

# --- Settings ---
base_dir = "/content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label"

input_img_dir = os.path.join(base_dir, "train_images")
input_txt_dir = os.path.join(base_dir, "train_labels")
output_dir = os.path.join(base_dir, "real2.0_blend100_sta")

#prompt = "Restore in stainless steel surface under cinematic lighting"
prompt = "In a dark natural lighting, low-lit environment"
#prompt = "In a bright natural lighting, well-lit environment"

# --- Create output directory ---
os.makedirs(output_dir, exist_ok=True)

# --- Device ---
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

# --- Load Stable Diffusion 1.5 Inpainting ---
print("⏳ Loading Stable Diffusion 1.5 Inpainting...")
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=dtype
).to(device)
print("✅ Model loaded successfully!")

# --- Get all images ---
image_files = [f for f in os.listdir(input_img_dir) if f.lower().endswith(".jpg")]
print(f"🔍 Found {len(image_files)} images.")

# --- Process loop ---
for idx, image_name in enumerate(sorted(image_files), start=1):
    img_path = os.path.join(input_img_dir, image_name)
    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(input_txt_dir, label_name)

    if not os.path.exists(label_path):
        print(f"⚠️ Skipping {image_name}: label not found.")
        continue

    # --- Load image ---
    image = Image.open(img_path).convert("RGB").resize((512, 512))

    # --- Create mask from YOLO bbox ---
    mask = Image.new("L", (512, 512), color=255)
    draw = ImageDraw.Draw(mask)

    with open(label_path, "r") as f:
        for line in f:
            cls, x_c, y_c, w, h = map(float, line.strip().split())

            x = (x_c - w / 2) * 512
            y = (y_c - h / 2) * 512
            w *= 512
            h *= 512

            draw.rectangle([x, y, x + w, y + h], fill=0)  # Black = inpaint region

    # --- Inpainting ---
    result = pipe(
        prompt=prompt,
        image=image,
        mask_image=mask,
        guidance_scale=7.5,
        strength=1.0,
    ).images[0]

    # --- Save image ---
    output_img_path = os.path.join(output_dir, "in" + image_name)
    result.save(output_img_path)

    # --- Copy YOLO label ---
    output_txt_path = os.path.splitext(output_img_path)[0] + ".txt"
    shutil.copy(label_path, output_txt_path)

    print(f"[{idx}/{len(image_files)}] ✅ Saved: {output_img_path}")

print("🎉 All batch inpainting completed! Output folder:", output_dir)


⏳ Loading Stable Diffusion 1.5 Inpainting...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


✅ Model loaded successfully!
🔍 Found 2 images.


  0%|          | 0/50 [00:00<?, ?it/s]

[1/2] ✅ Saved: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/real2.0_blend100_sta/in62.jpg


  0%|          | 0/50 [00:00<?, ?it/s]

[2/2] ✅ Saved: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/real2.0_blend100_sta/in63.jpg
🎉 All batch inpainting completed! Output folder: /content/drive/MyDrive/CV and YOLO result2025/Test_result_YOLO25/Prep_for_conf_24May25/image_n_label/real2.0_blend100_sta
